# AI Investment Research Assistant

The instructions below guide you through the process of creating a supervisor agent and subagents for an investment research assistant. Each section explains the purpose of the code cells that follow.

### PREREQUISITES:

Follow instructions on README.md to deploy web search stack, stock data stack, portfolio-optimization stack, and bda stack.

### Step 1: Environment Setup and Imports
In this first section, we import the required libraries and initialize the environment. 

In [ ]:
!pip install -q -r /home/sagemaker-user/amazon-bedrock-agent-samples/src/requirements.txt;

### Importing helper functions

On following section, we're adding `bedrock_agent_helper.py` and `knowledge_base_helper` on Python path, so the files can be recognized and their functionalities can be invoked.

Now, you're going to import from helper classes `bedrock_agent_helper.py` and `knowledge_base_helper.py`.

All interactions with Bedrock will be handled by these classes.

Following are methods that you're going to invoke on this notebook:

On `agents.py`:
- `create_agent`: Create a new agent and respective IAM roles
- `invoke`: Execute agent

On `knowledge_bases.py`:
- `create_or_retrieve_knowledge_base`: Create Knowledge Base on Amazon Bedrock if it doesn't exist or get info about previous created.
- `synchronize_data`: Read files on S3, convert text info into vectors and add that information on Vector Database.

In [ ]:
import sys
import argparse
from pathlib import Path

import boto3
import botocore

# Adjust the root path to go up 2 levels from the current working directory
ROOT_PATH = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT_PATH))  # Insert at the beginning of sys.path

# Importing custom modules
from src.utils.bedrock_agent import (
    Agent,
    SupervisorAgent,
    Task,
    Guardrail,
    region,
    account_id,
    agents_helper,
)
from src.utils.knowledge_base_helper import KnowledgeBasesForAmazonBedrock

# Initialize the Knowledge Base helper
kb_helper = KnowledgeBasesForAmazonBedrock()

# Initialize boto3 client
bedrock_client = boto3.client("bedrock")
#LLM = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
#LLM = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
LLM = "amazon.nova-lite-v1:0"

If running this code on a Sagemaker notebook, run the following cell and add inlinePolicy.json to the execution role printed out below. Replace execution role and account ID.

In [ ]:
sagemaker_client = boto3.client("sagemaker")
from sagemaker import get_execution_role

# Get the execution role name
execution_role_arn = get_execution_role()
print(execution_role_arn)

### Set up functions to clean up agents and create a guardrail
Function to delete agents and guardrails if they are already created

In [ ]:
def clean_up_agents():
    agents_helper.delete_agent(agent_name="investment_research_assistant", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="news_agent", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="quantitative_analysis_agent", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="smart_summarizer_agent", delete_role_flag=True, verbose=True)
    response = bedrock_client.list_guardrails()
    for _gr in response["guardrails"]:
        if _gr["name"] == "no_bitcoin_guardrail":
            print(f"Found guardrail: {_gr['id']}")
            guardrail_identifier = _gr["id"]
            bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_identifier)

Define Guardrail

In [ ]:
def create_guardrail():
    return Guardrail(
        "no_bitcoin_guardrail",
        "bitcoin_topic",
        "No Bitcoin or cryptocurrency allowed in the analysis.",
        denied_topics=["bitcoin", "crypto", "cryptocurrency"],
        blocked_input_response="Sorry, this agent cannot discuss bitcoin.",
        verbose=True,
    )

Delete old agents and guardrail if they exist, create new guardrail

In [ ]:
clean_up_agents()
Agent.set_force_recreate_default(True)
no_bitcoin_guardrail = create_guardrail()

### Create subagents
Create smart_summarizer_agent subagent

In [ ]:
smart_summarizer_agent = Agent.create(
            name="smart_summarizer_agent",
            role="A financial analyst specializing in synthesizing stock market trends and financial news into structured investment insights. The agent produces fact-based summaries to support strategic decision-making.",
            goal="Analyze stock trends and market news to generate insights.",
            instructions="""You are a Financial Analyst, responsible for analyzing stock trends and financial news to generate structured insights.
                            Combine stock price trends with financial news to identify key patterns.
                            Use your expertise to analyze macroeconomic indicators, company earnings, and market sentiment.
                            Ensure responses are fact-driven, clearly structured, and cite sources where applicable.
                            Do not generate financial advice—your role is to analyze and summarize available data objectively.
                            Keep analyses concise and insightful, focusing on major trends and anomalies.
                            **If given portfolio optimization pecentages, indicate that these are based on logic/math from the portfolio optimization tool, and are not considered financial advice**""",
            llm = LLM,
        )

Create quantitative_analysis subagent

In [ ]:
# Define Lambda ARNs
stock_data_lookup_arn = f"arn:aws:lambda:{region}:{account_id}:function:stock_data_lookup"
portfolio_optimization_arn = f"arn:aws:lambda:{region}:{account_id}:function:FSI-PortfolioTool-BedrockAgent"

quantitative_analysis_agent = Agent.create(
    name="quantitative_analysis_agent",
    role="Financial Data Collector",
    goal="Retrieve real-time and historic stock prices as well as optimizing a portfolio given tickers.",
    instructions="""You are a Stock Data and Portfolio Optimization Specialist. Your role is to retrieve real-time stock data and optimize investment portfolios.

Your capabilities include:
1. Retrieving stock price data using the `stock_data_lookup` tool.
2. Performing portfolio optimization when at least three stock tickers are provided.
3. Enforcing the portfolio optimization rule: If fewer than three tickers are provided, inform the user that optimization requires at least three.

Core behaviors:
- Always retrieve stock data from `stock_data_lookup` before running portfolio optimization.
- If portfolio optimization is requested, invoke `portfolio_optimization_action_group` only after retrieving stock data.
- Do not attempt to interpret financial trends—focus solely on data retrieval and portfolio structuring.
""",
    tools=[
        # Stock Data Lookup Tool
        {
            "code": stock_data_lookup_arn,
            "definition": {
                "name": "stock_data_lookup",
                "description": "Gets the 1-month stock price history for a given stock ticker, formatted as JSON.",
                "parameters": {
                    "ticker": {"description": "The ticker to retrieve price history for", "type": "string", "required": True}
                },
            },
        },
        # Portfolio Optimization Tool
        {
            "code": portfolio_optimization_arn,
            "definition": {
                "name": "portfolio_optimization",
                "description": "Optimizes a stock portfolio given a list of tickers and historical prices from the stock_data_lookup function.",
                "parameters": {
                    "tickers": {
                        "description": "A comma-separated list of stock tickers to include in the portfolio",
                        "type": "string",
                        "required": True
                    },
                    "prices": {
                        "description": "A JSON object with dates as keys and stock prices as values",
                        "type": "string",
                        "required": True
                    }
                }
            },
        }
    ],
    llm=LLM,
)


Create a knowledge base, create news_agent subagent, load the knowledge base"

In [ ]:
kb_name = "financial-analysis-kb" #change to your name
kb_description = "Access this knowledge base when needing to look up financial information like 10K reports, revenues, sales, net sales, loss and risks. Contains earnings calls."
kb_s3_bucket = "[your output bucket]"  #put your own output bucket name from bda stack
kb_id, ds_id = kb_helper.create_or_retrieve_knowledge_base(
    kb_name=kb_name,
    kb_description=kb_description,
    data_bucket_name=kb_s3_bucket,
    embedding_model="amazon.titan-embed-text-v2:0",
)

# Define Web Search Lambda ARN
web_search_arn = f"arn:aws:lambda:{region}:{account_id}:function:web_search"


news_agent = Agent.create(
    name="news_agent",
    role="Market News Researcher",
    goal="Fetch from the knowledge base. Then if needed, fetch latest relevant news for a given stock based on a ticker.",
    instructions=f"""You are a Financial Document & News Analyst responsible for extracting structured insights from official financial reports and real-time news.

Your capabilities include:
1. Extracting insights from earnings calls, SEC filings (10-K, 10-Q), and corporate press releases stored in the knowledge base (ID: {kb_id}).
2. Summarizing financial reports with a focus on factual accuracy.
3. Retrieving the latest financial news only **if the knowledge base lacks relevant information**.

Core behaviors:
- **Always check the knowledge base (ID: {kb_id}) first** before fetching external news.
- **Avoid unnecessary web searches**—use external news sources only if the knowledge base lacks sufficient information.
- Ensure all findings are **fact-based, neutral, and structured** for investment research.
""",
    tools=[
        {"code": web_search_arn, "definition": {
            "name": "web_search",
            "description": "Searches the web for investment news and earnings reports.",
            "parameters": {
                "search_query": {"description": "The query to search the web with", "type": "string", "required": True},
                "target_website": {"description": "Specific website to search", "type": "string", "required": False},
                "topic": {"description": "The topic being searched, such as 'news'", "type": "string", "required": False},
                "days": {"description": "Number of days of history to search", "type": "string", "required": False},
            },
        },
        },
         ],
    kb_id=kb_id,
    llm=LLM,
)

kb_helper.synchronize_data(kb_id, ds_id)

### Create the supervisor agent

In [ ]:
investment_research_assistant = SupervisorAgent.create(
    "investment_research_assistant",
    role="Investment Research Assistant",
    goal="A seasoned investment research expert responsible for orchestrating subagents to conduct a comprehensive stock analysis. This agent synthesizes market news, stock data, and smart_summarizer insights into a structured investment report.",
    collaboration_type="SUPERVISOR",
    instructions=f"""You are an Investment Research Assistant, responsible for overseeing and synthesizing financial research from specialized agents. Your role is to coordinate subagents to produce structured investment insights.

Your capabilities include:
1. Managing collaboration between subagents to retrieve and analyze financial data.
2. Synthesizing stock trends, financial reports, and market news into a structured analysis.
3. Delivering well-organized, fact-based investment insights with clear distinctions between data sources.

Available subagents:
- **news_agent**: Retrieves and summarizes the latest financial news.  
  - **Always instruct news_agent to check the knowledge base (ID: {kb_id}) first before using external web searches**.
- **quantitative_analysis_agent**: Provides real-time and historical stock prices.  
  - For portfolio optimization, retrieve stock data via `stock_data_lookup` before calling `portfolio_optimization_action_group`.
- **smart_summarizer_agent**: Synthesizes financial data and market trends into a structured investment insight.

Core behaviors:
- Only invoke a subagent when necessary. Do not invoke agent for information not requested by user.
- Ensure responses are **well-structured, clearly formatted, and relevant to investor decision-making**.
- Differentiate between financial news, technical stock analysis, and synthesized insights.
""",
    collaborator_agents=[
        {
            "agent": "news_agent",
            "instructions": f"Always check the knowledge base (ID: {kb_id}) first. Use this collaborator for finding news and analyzing specific documents."
        },
        {
            "agent": "quantitative_analysis_agent",
            "instructions": "Use this collaborator for retrieving stock price history and performing portfolio optimization."
        },
        {
            "agent": "smart_summarizer_agent",
            "instructions": "Use this collaborator for synthesizing stock trends, financial data, and generating structured investment insights."
        }
    ],
    collaborator_objects=[news_agent, quantitative_analysis_agent, smart_summarizer_agent],
    #guardrail=no_bitcoin_guardrail,
    llm=LLM,
    #verbose=False,
)


### Example queries to the supervisor agent

In [ ]:
request = "what's AMZN stock price doing over the last week and relate that to recent news"
print(f"Request:\n{request}\n")
trace_level = "core"
result = investment_research_assistant.invoke(
    request,
    enable_trace=True,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

In [ ]:
request = "Optimize my portfolio with AMZN, MSFT, and GOOG"
print(f"Request:\n{request}\n")
trace_level = "core"
result = investment_research_assistant.invoke(
    request,
    enable_trace=True,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

In [ ]:
request = "Tell me about 2023 Q1 amazon earnings call."
print(f"Request:\n{request}\n")
trace_level = "core"
result = investment_research_assistant.invoke(
    request,
    enable_trace=False,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

In [ ]:
request = "Analyze Amazon’s financial health based on the 2024 10k report. Calculate important financial ratios. Limit to 5 sentences"
print(f"Request:\n{request}\n")
trace_level = "outline"
result = investment_research_assistant.invoke(
    request,
    enable_trace=False,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

In [ ]:
request = "Tell me about bitcoin"
print(f"Request:\n{request}\n")
trace_level = "outline"
result = investment_research_assistant.invoke(
    request,
    enable_trace=False,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")